# MAMMQA_Provided_Context_16

**Experiment:** `MAMMQA_Provided_Context_16_seed42`  
**Role:** provided-context / reasoning-only control  
**Not a retrieval experiment:** no BM25, dense/CLIP retrieval, router, graph, verifier, feedback, or web access.

## Audit of the attached notebook

**REUSED**

- Official MultiModalQA dev/text/table/image loading and metadata context IDs.
- One memory-safe 4-bit `Qwen/Qwen2.5-VL-7B-Instruct` backend.
- MAMMQA Stage I modality experts, Stage II anchor-specific synthesis, and Stage III aggregation.
- Paper-consistent sampling: temperature 0.3, top-p 0.7, sampling enabled.

**FIXED**

- First-20 selection is replaced by the exact RAG smoke qids or its deterministic seed-42 stratified fallback.
- Stage I/II agents are activated only for genuinely available modalities.
- Stage II receives the anchor expert output plus raw *other-modality* evidence.
- All valid gold variants are evaluated with the same normalization and EM/F1 implementation as the RAG notebook.
- Gold/evaluation annotations are unavailable until a complete prediction exists.

**REMOVED**

- Wikipedia/image downloads, image titles as visual evidence, Torch/CUDA reinstalls, zero-shot/CoT/ToT runs,
  robustness runs, question-agnostic ablations, and first-answer-only evaluation.

**ADDED**

- Official image-ID-to-local-pixel resolution, strict visual sanity checks, resumability scoped to this experiment,
  complete reasoning traces, failure analysis, exact output contracts, and an optional read-only RAG comparison.

This 16-question run is a controlled smoke test, not a dataset-wide benchmark.

In [ ]:
# Cell 1 - environment diagnostics and missing-package-only installation
import importlib.util, platform, subprocess, sys

print("Python:",sys.version)
try:
    import torch
    print("PyTorch:",torch.__version__,"CUDA:",torch.cuda.is_available(),"GPUs:",torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):print(i,torch.cuda.get_device_name(i))
except Exception as e:print("PyTorch diagnostic failed:",e)

# Never replace Kaggle's Torch, CUDA, or torchvision stack.
REQUIRED={
    "transformers":"transformers>=4.49,<5", "accelerate":"accelerate>=1,<2",
    "bitsandbytes":"bitsandbytes>=0.45,<1", "qwen_vl_utils":"qwen-vl-utils>=0.0.8",
    "pandas":"pandas>=2,<3", "tabulate":"tabulate>=0.9,<1",
}
missing=[spec for module,spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing only missing packages:",missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
else:print("All required packages are already installed; Torch/CUDA stack left unchanged.")

In [ ]:
# Cell 2 - fixed experiment configuration, imports, seed and isolated outputs
import ast, gc, hashlib, json, math, os, random, re, string, time, zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import numpy as np, pandas as pd, torch
from PIL import Image

EXPERIMENT_ID="MAMMQA_Provided_Context_16_seed42"
SEED=42
MODEL_NAME="Qwen/Qwen2.5-VL-7B-Instruct"
CONTEXT_MODE="paper_context"
TEMPERATURE=0.3; TOP_P=0.7; DO_SAMPLE=True
MAX_NEW_TOKENS=256; MAX_INPUT_TOKENS=6144
MAX_TEXT_CHARS_PER_DOC=8000; MAX_TOTAL_TEXT_CHARS=18000; MAX_TABLE_CHARS=18000
IMAGE_REASONING_USES_TITLE=False; ENABLE_WEB=False
PROMPT_VERSION="MAMMQA_Appendix_C_Qwen_local_v1"
ANSWER_PARSER_VERSION="shared_rag_parse_json_object_v1+conservative_tags_v1"
EVALUATOR_VERSION="shared_rag_all_gold_em_f1_v1"

OUTPUT_ROOT=Path("/kaggle/working/mammqa_provided_context_16")
DIRS={name:OUTPUT_ROOT/name for name in ["predictions","metrics","logs","official_images"]}
for p in DIRS.values():p.mkdir(parents=True,exist_ok=True)
PREDICTION_PATH=DIRS["predictions"]/"MAMMQA_Provided_Context_16.jsonl"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert CONTEXT_MODE=="paper_context" and not ENABLE_WEB and IMAGE_REASONING_USES_TITLE is False

def gpu_memory(label):
    if not torch.cuda.is_available():print(label,"CPU only"); return
    vals=[]
    for i in range(torch.cuda.device_count()):
        free,total=torch.cuda.mem_get_info(i); vals.append(f"GPU{i} free={free/2**30:.2f}/{total/2**30:.2f} GiB")
    print(label," | ".join(vals))

print("PROVIDED-CONTEXT BASELINE")
print("REUSED: dataset loaders, local Qwen backend, and MAMMQA three-stage reasoning")
print("FIXED: selection, official pixels, dynamic agents, leakage boundary, parser/evaluator, resumability")
print("REMOVED: Wikipedia/web images, first-20 run, unavailable agents, retrieval/verifier/feedback, unrelated ablations")
print("ADDED: strict sanity checks, full traces, failure analysis, isolated outputs")
gpu_memory("startup:")

In [ ]:
# Cell 3 - confirmed official MultiModalQA dataset paths
DATA_ROOT=Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset")
NAMES={
 "dev":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl",
 "texts":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl",
 "tables":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl",
 "images":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl"}
FOLDERS={k:f"MMQA_{k}.jsonl" for k in NAMES}

def resolve_dataset_file(key):
    primary=DATA_ROOT/FOLDERS[key]/NAMES[key]
    if primary.is_file():return primary
    roots=[DATA_ROOT,Path("/kaggle/input")]
    matches=sorted({p.resolve() for root in roots if root.exists() for p in root.rglob(NAMES[key])})
    if len(matches)!=1:raise FileNotFoundError(f"{key}: expected one {NAMES[key]}, found {matches}")
    return matches[0]

PATHS={k:resolve_dataset_file(k) for k in NAMES}
for k,p in PATHS.items():print(f"{k:7s} {p}")

In [ ]:
# Cell 4 - strict JSONL loading and complete provided-context lookups
def load_jsonl(path):
    rows=[]
    with Path(path).open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try:rows.append(json.loads(line))
                except json.JSONDecodeError as e:raise ValueError(f"{path}:{no}: {e}") from e
    return rows

dev_raw=load_jsonl(PATHS["dev"]); texts_raw=load_jsonl(PATHS["texts"])
tables_raw=load_jsonl(PATHS["tables"]); images_raw=load_jsonl(PATHS["images"])
TEXT_BY_ID={str(r["id"]):r for r in texts_raw}
TABLE_BY_ID={str(r["id"]):r for r in tables_raw}
IMAGE_BY_ID={str(r["id"]):r for r in images_raw}
assert len(dev_raw)>2400 and len(texts_raw)>200000 and len(tables_raw)>10000 and len(images_raw)>50000
print(f"Dataset: {len(dev_raw):,} dev questions | {len(texts_raw):,} texts | {len(tables_raw):,} original tables | {len(images_raw):,} image records")

In [ ]:
# Cell 5 - official image discovery and pixel validation (no URLs, no web fallback)
def safe_extract(zp,dst):
    with zipfile.ZipFile(zp) as z:
        root=dst.resolve()
        for member in z.infolist():
            target=(dst/member.filename).resolve()
            if root not in target.parents and target!=root:raise ValueError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(dst)

def discover_image_paths(records):
    expected={Path(str(r.get("path",""))).name for r in records if r.get("path")}
    mounts=Path("/kaggle/input"); extract=DIRS["official_images"]/"extracted"
    named=[] if not mounts.exists() else [p for p in mounts.rglob("final_dataset_images") if p.is_dir()]
    zips=[] if not mounts.exists() else [p for p in mounts.rglob("*.zip") if re.search(r"(mmqa|multimodal).*image|image.*(mmqa|multimodal)",p.name,re.I)]
    valid=[]
    for zp in zips:
        try:
            with zipfile.ZipFile(zp) as zh:
                if any(Path(n).name in expected for n in zh.namelist()):valid.append(zp)
        except zipfile.BadZipFile:pass
    if not named and valid:
        if len(valid)>1:raise RuntimeError(f"Ambiguous official image ZIPs: {valid}")
        marker=extract/".complete"
        if not marker.exists():
            extract.mkdir(parents=True,exist_ok=True); safe_extract(valid[0],extract); marker.touch()
        named=[extract]
    roots=named or ([mounts] if mounts.exists() else [])
    by_name={}
    for root in roots:
        for p in root.rglob("*"):
            if p.is_file() and p.name in expected:
                if p.name in by_name and by_name[p.name]!=p:raise RuntimeError(f"Ambiguous official image filename: {p.name}")
                by_name[p.name]=p
    return {str(r["id"]):by_name[Path(str(r.get("path",""))).name] for r in records if Path(str(r.get("path",""))).name in by_name}

IMAGE_ID_TO_PATH=discover_image_paths(images_raw)
IMAGE_PATH_TO_ID={str(path):image_id for image_id,path in IMAGE_ID_TO_PATH.items()}
missing=len(images_raw)-len(IMAGE_ID_TO_PATH)
print(f"total image metadata records={len(images_raw):,}")
print(f"resolved official images={len(IMAGE_ID_TO_PATH):,}")
print(f"missing images={missing:,}")
print(f"resolution percentage={100*len(IMAGE_ID_TO_PATH)/max(1,len(images_raw)):.2f}%")
assert len(IMAGE_ID_TO_PATH)==len(images_raw),"STOP: not every official MMQA image resolved"
for p in list(IMAGE_ID_TO_PATH.values())[:10]:
    with Image.open(p) as im:im.verify()
print("Official PIL pixel validation passed")

In [ ]:
# Cell 6 - explicit inference/evaluation separation and gold parsing
@dataclass(frozen=True)
class InferenceInput:
    qid:str; question:str; context_mode:str
    provided_text_ids:Tuple[str,...]; provided_table_ids:Tuple[str,...]; provided_image_ids:Tuple[str,...]
    text_records:Tuple[dict,...]; table_records:Tuple[dict,...]; image_records:Tuple[dict,...]
    available_modalities:Tuple[str,...]

@dataclass(frozen=True)
class EvaluationOnly:
    gold_answers:Tuple[str,...]; gold_support_ids:Tuple[str,...]; gold_support_modalities:Dict[str,str]

FORBIDDEN_INFERENCE_FIELDS={"answers","gold_answers","supporting_context","gold_support_ids","intermediate_answers","type","modalities"}
def assert_safe_inference(x):
    assert isinstance(x,InferenceInput)
    assert not (set(asdict(x))&FORBIDDEN_INFERENCE_FIELDS)
    assert x.context_mode=="paper_context"

def answer_strings(row):
    vals=[]
    def walk(x):
        if isinstance(x,str):vals.append(x)
        elif isinstance(x,list):
            for y in x:walk(y)
        elif isinstance(x,dict):
            for k in ["answer","answers","aliases","normalized_answer"]:
                if k in x:walk(x[k])
    walk(row.get("answers",[])); return tuple(dict.fromkeys(v.strip() for v in vals if v.strip()))

def support_info(row):
    ids=[]; mods={}; known={"text":set(TEXT_BY_ID),"table":set(TABLE_BY_ID),"image":set(IMAGE_BY_ID)}
    def add(v,mod=None):
        s=str(v)
        if s in {"","None"}:return
        inferred=mod or next((m for m,z in known.items() if s in z),None)
        if inferred and s in known[inferred]:ids.append(s); mods[s]=inferred
    def walk(x,mod=None):
        if isinstance(x,list):
            for y in x:walk(y,mod)
        elif isinstance(x,dict):
            local=x.get("modality",x.get("doc_part",mod)); local={"text":"text","table":"table","image":"image"}.get(str(local).lower(),mod)
            for k,v in x.items():
                km={"text_doc_id":"text","table_id":"table","image_doc_id":"image","doc_id":local}.get(k)
                if km:add(v,km)
                elif k not in {"answer","text","content"}:walk(v,local)
        elif mod:add(x,mod)
    walk(row.get("supporting_context",[])); return tuple(dict.fromkeys(ids)),mods

def evaluation_only(row):
    ids,mods=support_info(row)
    return EvaluationOnly(answer_strings(row),ids,mods)

In [ ]:
# Cell 7 - exact RAG qids when available, otherwise identical deterministic stratified fallback
DEV_BY_QID={str(r.get("qid",r.get("id",""))):r for r in dev_raw}

def qids_from_selection_file(path):
    payload=json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(payload,dict):payload=payload.get("questions",payload.get("selected_questions",[]))
    qids=[]
    for x in payload:
        qids.append(str(x.get("qid",x.get("id",""))) if isinstance(x,dict) else str(x))
    return qids

def deterministic_stratified_16(rows):
    strata=defaultdict(list)
    for row in rows:
        ev=evaluation_only(row); combo="+".join(sorted(set(ev.gold_support_modalities.values()))) or "unknown"
        strata[(str(row.get("metadata",{}).get("type","unknown")),combo)].append(row)
    rng=random.Random(SEED)
    for values in strata.values():rng.shuffle(values)
    out=[]
    while len(out)<16:
        progressed=False
        for key in sorted(strata,key=str):
            if strata[key] and len(out)<16:out.append(strata[key].pop()); progressed=True
        if not progressed:break
    return [str(r.get("qid",r.get("id",""))) for r in out]

selection_candidates=[]
explicit=[
    Path("/kaggle/working/hm_mammqa/qa_smoke16_full_pipeline_seed42/selected_questions.json"),
    OUTPUT_ROOT/"selected_questions.json",
]
for p in explicit:
    if p.is_file():selection_candidates.append(p)
input_root=Path("/kaggle/input")
if input_root.exists():
    selection_candidates += [p for p in input_root.rglob("selected_questions.json") if p not in selection_candidates and any(token in str(p).lower() for token in ["hm_mammqa","qa_smoke16","full_pipeline"])]

valid_external=[]
for p in selection_candidates:
    try:
        qids=qids_from_selection_file(p)
        if len(qids)==16 and len(set(qids))==16 and set(qids)<=set(DEV_BY_QID):valid_external.append((p,qids))
    except Exception as e:print("Ignoring invalid selection file",p,e)

if valid_external:
    unique_sets={frozenset(qids) for _,qids in valid_external}
    if len(unique_sets)!=1:raise RuntimeError(f"Conflicting 16-qid selection files: {[str(p) for p,_ in valid_external]}")
    SELECTED_QIDS=valid_external[0][1]; SELECTION_SOURCE=str(valid_external[0][0]); RAG_QIDS=SELECTED_QIDS.copy()
else:
    SELECTED_QIDS=deterministic_stratified_16(dev_raw); SELECTION_SOURCE="deterministic_stratified_seed42_fallback"; RAG_QIDS=None

assert len(SELECTED_QIDS)==16
assert len(set(SELECTED_QIDS))==16
if RAG_QIDS is not None:assert SELECTED_QIDS==RAG_QIDS
SELECTED_RAW=[DEV_BY_QID[qid] for qid in SELECTED_QIDS]
(OUTPUT_ROOT/"selected_questions.json").write_text(json.dumps([{"qid":qid,"question":DEV_BY_QID[qid]["question"]} for qid in SELECTED_QIDS],indent=2,ensure_ascii=False),encoding="utf-8")
print("Selection source:",SELECTION_SOURCE)
print("Selected questions:",len(SELECTED_QIDS))
print("Unique qids:",len(set(SELECTED_QIDS)))
print("Selected qids:",SELECTED_QIDS)

In [ ]:
# Cell 8 - question-associated paper-context construction (never supporting_context)
def id_list(value):
    if value is None:return []
    if isinstance(value,(list,tuple,set)):return [str(x) for x in value if str(x) not in {"","None"}]
    return [str(value)] if str(value) not in {"","None"} else []

def cell_text(c):
    if isinstance(c,dict):return str(c.get("text",c.get("value","")))
    return str(c)

def table_parts(raw):
    tb=raw.get("table",raw); header=tb.get("header",[]); rows=tb.get("table_rows",tb.get("rows",[]))
    heads=[str(x.get("column_name",x.get("text",""))) if isinstance(x,dict) else str(x) for x in header]
    return heads,[[cell_text(c) for c in row] for row in rows]

def markdown_table(headers,rows):
    width=max([len(headers),*(len(r) for r in rows)] or [0]); headers=(headers+[f"col_{i}" for i in range(len(headers),width)])[:width]
    clean=lambda s:str(s).replace("|","\\|").replace("\n"," ")
    lines=["| "+" | ".join(map(clean,headers))+" |","| "+" | ".join(["---"]*width)+" |"]
    lines += ["| "+" | ".join(map(clean,row+[""]*(width-len(row))))+" |" for row in rows]
    return "\n".join(lines)

def build_inference_input(row):
    meta=row.get("metadata",{})
    text_ids=id_list(meta.get("text_doc_ids",[])); table_ids=id_list(meta.get("table_id",meta.get("table_ids",[]))); image_ids=id_list(meta.get("image_doc_ids",[]))
    missing_text=[x for x in text_ids if x not in TEXT_BY_ID]; missing_table=[x for x in table_ids if x not in TABLE_BY_ID]; missing_image=[x for x in image_ids if x not in IMAGE_ID_TO_PATH]
    assert not missing_text and not missing_table and not missing_image,f"Unresolved provided context IDs: text={missing_text}, table={missing_table}, image={missing_image}"
    texts=tuple({"id":x,"title":str(TEXT_BY_ID[x].get("title","")),"text":str(TEXT_BY_ID[x].get("text",TEXT_BY_ID[x].get("paragraph","")))} for x in text_ids)
    tables=[]
    for x in table_ids:
        h,rows=table_parts(TABLE_BY_ID[x]); tables.append({"id":x,"title":str(TABLE_BY_ID[x].get("title",TABLE_BY_ID[x].get("name",""))),"headers":h,"rows":rows,"markdown":markdown_table(h,rows)})
    images=tuple({"id":x,"path":str(IMAGE_ID_TO_PATH[x])} for x in image_ids)
    available=tuple(m for m,present in [("text",bool(texts)),("table",bool(tables)),("image",bool(images))] if present)
    item=InferenceInput(str(row.get("qid",row.get("id",""))),str(row["question"]),CONTEXT_MODE,tuple(text_ids),tuple(table_ids),tuple(image_ids),texts,tuple(tables),images,available)
    assert_safe_inference(item); return item

def pil_loadable(path):
    try:
        with Image.open(path) as im:im.verify()
        return True
    except Exception:return False

PROVIDED_INPUTS=[build_inference_input(r) for r in SELECTED_RAW]
for item in PROVIDED_INPUTS:
    image_ok=all(Path(r["path"]).is_file() and pil_loadable(r["path"]) for r in item.image_records)
    print({"qid":item.qid,"available_modalities":list(item.available_modalities),"official_image_resolved":(image_ok if item.image_records else "N/A"),"image_pixels_loadable":(image_ok if item.image_records else "N/A")})
    if item.image_records:assert image_ok,"STOP: provided image context exists but official pixels are not loadable"
assert any(x.image_records for x in PROVIDED_INPUTS),"The selected comparison set must include image context"
print("PROVIDED CONTEXT READY; supporting_context was not used to build model inputs")

In [ ]:
# Cell 9 - load Qwen2.5-VL once with the existing memory-safe 4-bit strategy
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
gpu_memory("before Qwen:")
processor=AutoProcessor.from_pretrained(MODEL_NAME,min_pixels=256*28*28,max_pixels=384*28*28)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map="auto",low_cpu_mem_usage=True).eval()
assert model is not None and processor is not None
print("QWEN LOADED")
for i in range(torch.cuda.device_count()):print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
gpu_memory("GPU memory information:")

In [ ]:
# Cell 10 - paper-aligned MAMMQA Prompt A/B/C templates
PROMPT_A='''You are an expert agent specialized in analyzing a single input modality (text, table, or image).

Task:
1. Identify the modality type.
2. Analyze only the supplied modality and extract relevant factual insights, including entities, numbers, and temporal information.
3. Break the original question into useful simpler subquestions.
4. Answer modality-local subquestions only when the supplied evidence supports them. Explicitly mark insufficient information instead of inventing an answer.

Return a structured response using:
<modality>...</modality>
<insights>...</insights>
<subanswer>...</subanswer>
<answer>...</answer>
Use only the provided data and no external knowledge.'''

PROMPT_B='''You are an expert cross-modal synthesis agent. The input contains the original question, the insight produced by one anchor-modality expert, and raw evidence from every other available modality.

Task:
1. Treat the named modality as the anchor and examine its expert insight.
2. Reconcile that insight with the raw non-anchor evidence.
3. Derive useful subanswers and resolve agreement or conflict across modalities.
4. Provide a final answer only when supported; otherwise state that the evidence is insufficient.

Return:
<insights>...</insights>
<subanswer>...</subanswer>
<answer>...</answer>
Use only the supplied evidence and no external knowledge.'''

PROMPT_C='''You are the final MAMMQA aggregator. You receive the original question and the outputs of the active anchor-specific synthesis agents, but no raw evidence.

Follow this hierarchy:
A. Consistency Check: when multiple synthesis agents agree with supported reasoning, prefer the consistent answer.
B. Fallback Rule: when some synthesis output is uncertain, rely on the best-supported available synthesis.
C. Conflict Resolution: when outputs disagree, compare their evidence-grounded reasoning and select the most strongly supported conclusion.
D. Final Synthesis: produce the final answer or explicitly abstain if the synthesis outputs remain insufficient.

You may briefly explain the adjudication, but finish with exactly one explicit field:
<answer>final answer</answer>
Do not use external knowledge.'''

print("MAMMQA prompts loaded:",PROMPT_VERSION)

In [ ]:
# Cell 11 - Qwen generation, shared RAG parser core and conservative answer extraction
QWEN_AUDIT=[]; IMAGE_DEBUG_PRINTED=False

def stable_call_seed(qid,stage):
    return (SEED+int(hashlib.sha256(f"{qid}:{stage}".encode()).hexdigest()[:8],16))%(2**31-1)

def run_vl_model(system_prompt,user_text,image_paths=None,max_new_tokens=MAX_NEW_TOKENS,qid="",stage=""):
    global IMAGE_DEBUG_PRINTED
    paths=[str(Path(p)) for p in (image_paths or []) if p and Path(p).is_file()]
    content=[{"type":"image","image":f"file://{p}"} for p in paths]+[{"type":"text","text":user_text}]
    messages=[{"role":"system","content":system_prompt},{"role":"user","content":content}]
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    imgs,vids=process_vision_info(messages)
    if paths:
        assert imgs and len(imgs)==len(paths),"Official pixels failed to enter the Qwen message"
        if not IMAGE_DEBUG_PRINTED:
            print("IMAGE PIXELS SENT TO QWEN: True")
            print("image_id =",IMAGE_PATH_TO_ID.get(paths[0],"UNKNOWN"))
            print("path exists =",Path(paths[0]).is_file())
            IMAGE_DEBUG_PRINTED=True
    kwargs={"text":[prompt],"return_tensors":"pt","padding":True,"truncation":True,"max_length":MAX_INPUT_TOKENS}
    if imgs:kwargs.update(images=imgs,videos=vids)
    inputs=processor(**kwargs).to(model.device)
    call_seed=stable_call_seed(qid,stage); random.seed(call_seed); np.random.seed(call_seed); torch.manual_seed(call_seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(call_seed)
    with torch.inference_mode():
        ids=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=DO_SAMPLE,temperature=TEMPERATURE,top_p=TOP_P,pad_token_id=processor.tokenizer.eos_token_id)
    raw=processor.batch_decode(ids[:,inputs.input_ids.shape[1]:],skip_special_tokens=True)[0].strip()
    QWEN_AUDIT.append({"qid":qid,"stage":stage,"image_count":len(paths),"image_paths":paths,"seed":call_seed})
    return raw

# Exact JSON-object parser core reused from the HM-RAG + MAMMQA notebook.
def parse_json_object(text):
    candidates=[text]
    candidates += re.findall(r"```(?:json)?\s*(\{.*?\})\s*```",text,re.S|re.I)
    start=text.find("{"); end=text.rfind("}")
    if start>=0 and end>start:candidates.append(text[start:end+1])
    for c in candidates:
        try:
            parsed=json.loads(c)
            if isinstance(parsed,dict):return parsed
            if isinstance(parsed,(str,int,float,bool)):
                value=str(parsed); return {"value":value,"candidate_answer":value,"best_answer":value,"answer":value}
        except Exception:pass
    return {}

def parse_final_answer(raw):
    text=str(raw or "").strip()
    matches=re.findall(r"<answer>\s*(.*?)\s*</answer>",text,re.S|re.I)
    if matches:return matches[-1].strip()
    obj=parse_json_object(text)
    for key in ["answer","final_answer","best_answer","candidate_answer","value"]:
        if str(obj.get(key,"")).strip():return str(obj[key]).strip()
    matches=re.findall(r"(?:final\s+answer|answer)\s*[:\-]\s*(.+)",text,re.I)
    if matches:return matches[-1].strip().strip("`*_ ")
    lines=[x.strip().strip("`*_ ") for x in text.splitlines() if x.strip()]
    return lines[-1] if lines else text

print("Generation:",{"temperature":TEMPERATURE,"top_p":TOP_P,"do_sample":DO_SAMPLE})
print("Answer parser:",ANSWER_PARSER_VERSION)

In [ ]:
# Cell 12 - dynamic Stage I, anchor-specific Stage II, and evidence-free Stage III inputs
def bounded(text,limit):
    text=str(text or "")
    return text if len(text)<=limit else text[:limit]+"\n[deterministic character cap reached]"

def format_text_records(records):
    blocks=[]
    for r in records:blocks.append(f"[TEXT ID: {r['id']}]\nTitle: {r['title']}\n{bounded(r['text'],MAX_TEXT_CHARS_PER_DOC)}")
    return bounded("\n\n".join(blocks),MAX_TOTAL_TEXT_CHARS)

def format_table_records(records):
    return bounded("\n\n".join(f"[TABLE ID: {r['id']}]\nTitle: {r['title']}\n{r['markdown']}" for r in records),MAX_TABLE_CHARS)

def make_stage1_input(item,modality):
    assert modality in item.available_modalities
    if modality=="text":return f"Question: {item.question}\n\nRaw text evidence:\n{format_text_records(item.text_records)}",[]
    if modality=="table":return f"Question: {item.question}\n\nRaw table evidence:\n{format_table_records(item.table_records)}",[]
    paths=[r["path"] for r in item.image_records]
    opaque_ids=[r["id"] for r in item.image_records]
    return f"Question: {item.question}\n\nOfficial image pixels are attached for opaque evidence IDs: {opaque_ids}. No image titles are supplied.",paths

def make_stage2_input(item,anchor,anchor_output):
    assert anchor in item.available_modalities
    sections=[f"Question: {item.question}",f"Anchor modality: {anchor}",f"Stage-I {anchor} expert output:\n{anchor_output}"]
    image_paths=[]; raw_non_anchor=[]
    for mod in item.available_modalities:
        if mod==anchor:continue
        raw_non_anchor.append(mod)
        if mod=="text":sections.append("Raw non-anchor text evidence:\n"+format_text_records(item.text_records))
        elif mod=="table":sections.append("Raw non-anchor table evidence:\n"+format_table_records(item.table_records))
        elif mod=="image":
            sections.append(f"Raw non-anchor official image pixels attached for opaque IDs: {[r['id'] for r in item.image_records]}")
            image_paths=[r["path"] for r in item.image_records]
    return "\n\n".join(sections),image_paths,raw_non_anchor

def make_aggregator_input(question,syntheses):
    payload={"question":question,"synthesis_outputs":syntheses}
    return json.dumps(payload,ensure_ascii=False)

def modality_expert(item,modality):
    user,paths=make_stage1_input(item,modality)
    return run_vl_model(PROMPT_A,user,paths,qid=item.qid,stage=f"stage1_{modality}")

def anchor_synthesizer(item,anchor,anchor_output):
    user,paths,_=make_stage2_input(item,anchor,anchor_output)
    return run_vl_model(PROMPT_B,user,paths,qid=item.qid,stage=f"stage2_{anchor}_anchor")

def aggregator(item,syntheses):
    # Intentionally only question + Stage-II outputs; no raw evidence or gold.
    return run_vl_model(PROMPT_C,make_aggregator_input(item.question,syntheses),qid=item.qid,stage="stage3_aggregator")

In [ ]:
# Cell 13 - complete reasoning-only MAMMQA pipeline
def run_mammqa(item,verbose=True):
    assert_safe_inference(item); audit_start=len(QWEN_AUDIT); started=time.perf_counter()
    modalities=list(item.available_modalities)
    assert modalities,"At least one provided modality is required"
    if verbose:print(f"\n{item.qid}: modalities={modalities}")
    stage1={}
    for mod in modalities:
        stage1[mod]=modality_expert(item,mod)
        if verbose:print("Stage I",mod,"complete")
    stage2={}; stage2_input_audit={}
    for anchor in modalities:
        user,paths,raw_other=make_stage2_input(item,anchor,stage1[anchor])
        stage2_input_audit[anchor]={"anchor_expert_used":anchor,"raw_non_anchor_modalities":raw_other,"official_image_pixel_count":len(paths)}
        stage2[anchor]=run_vl_model(PROMPT_B,user,paths,qid=item.qid,stage=f"stage2_{anchor}_anchor")
        if verbose:print("Stage II",anchor,"anchor complete")
    raw_aggregator=aggregator(item,stage2); final_answer=parse_final_answer(raw_aggregator)
    events=QWEN_AUDIT[audit_start:]; expected_agents=2*len(modalities)+1
    assert len(events)==expected_agents
    return {
        "qid":item.qid,"question":item.question,"context_mode":item.context_mode,
        "available_modalities":modalities,"provided_text_ids":list(item.provided_text_ids),"provided_table_ids":list(item.provided_table_ids),"provided_image_ids":list(item.provided_image_ids),
        "context_source_counts":{"text":len(item.text_records),"table":len(item.table_records),"image":len(item.image_records)},
        "resolved_official_image_paths":[r["path"] for r in item.image_records],
        "stage1":stage1,"stage2":stage2,"stage2_input_audit":stage2_input_audit,
        "aggregator_input_keys":["question","synthesis_outputs"],"raw_aggregator_output":raw_aggregator,"parsed_final_answer":final_answer,
        "active_agent_count":expected_agents,"abstain":norm_answer(final_answer) in {"abstain","insufficient","insufficient information"},
        "runtime_trace":{"qwen_stages":[e["stage"] for e in events],"image_pixel_count":sum(e["image_count"] for e in events),"image_pixel_paths":list(dict.fromkeys(p for e in events for p in e["image_paths"]))},
        "latency_seconds":time.perf_counter()-started,
    }

In [ ]:
# Cell 14 - identical RAG normalization and all-gold EM/F1 evaluator
ARTICLES={"a","an","the"}
def norm_answer(s):
    s=str(s).lower(); s="".join(" " if c in string.punctuation else c for c in s)
    return " ".join(w for w in s.split() if w not in ARTICLES)

def em_f1(pred,golds):
    p=norm_answer(pred).split(); best_em=0; best_f1=0.0
    for g0 in golds:
        g=norm_answer(g0).split(); best_em=max(best_em,int(p==g)); common=Counter(p)&Counter(g); n=sum(common.values())
        f=0 if not p or not g or n==0 else 2*(n/len(p))*(n/len(g))/((n/len(p))+(n/len(g)))
        best_f1=max(best_f1,f)
    return best_em,best_f1

assert em_f1("The Paris!",["London","Paris"])==(1,1.0)
assert em_f1("red car",["car"])[0]==0 and em_f1("red car",["car"])[1]>0
print("All-gold evaluator synthetic tests passed:",EVALUATOR_VERSION)

In [ ]:
# Cell 15 - required pre-run scientific sanity tests
SANITY={}
SANITY["test_1_exactly_16_unique_qids"]=len(SELECTED_QIDS)==len(set(SELECTED_QIDS))==16
SANITY["test_2_same_as_rag_if_present"]=(RAG_QIDS is None or SELECTED_QIDS==RAG_QIDS)
SANITY["test_3_official_images_complete"]=(len(IMAGE_ID_TO_PATH)==len(images_raw) and all(pil_loadable(r["path"]) for x in PROVIDED_INPUTS for r in x.image_records))
SANITY["test_4_no_wikipedia_fetch"]=(not ENABLE_WEB and "fetch_wiki_image" not in globals() and "download_image" not in globals())
image_item=next(x for x in PROVIDED_INPUTS if "image" in x.available_modalities)
_,image_paths=make_stage1_input(image_item,"image")
SANITY["test_5_image_expert_gets_pixels"]=bool(image_paths) and all(Path(p).is_file() and pil_loadable(p) for p in image_paths)
SANITY["test_6_dynamic_agents"]=(2*3+1==7 and 2*2+1==5 and all((2*len(x.available_modalities)+1) in {3,5,7} for x in PROVIDED_INPUTS))
probe=next((x for x in PROVIDED_INPUTS if len(x.available_modalities)>=2),None)
probe_anchor=probe.available_modalities[0]; probe_user,probe_paths,probe_raw=make_stage2_input(probe,probe_anchor,"ANCHOR_INSIGHT_SENTINEL")
SANITY["test_7_stage2_anchor_plus_raw_other"]=("ANCHOR_INSIGHT_SENTINEL" in probe_user and set(probe_raw)==set(probe.available_modalities)-{probe_anchor})
agg_probe=json.loads(make_aggregator_input("Q",{"text":"S"}))
SANITY["test_8_aggregator_no_raw_evidence"]=(set(agg_probe)=={"question","synthesis_outputs"})
SANITY["test_9_gold_hidden_until_prediction"]=(not (set(asdict(PROVIDED_INPUTS[0]))&FORBIDDEN_INFERENCE_FIELDS))
SANITY["test_10_all_gold_evaluator"]=(em_f1("answer b",["answer a","answer b"])[0]==1)
SANITY["test_11_shared_parser_evaluator"]=(ANSWER_PARSER_VERSION.startswith("shared_rag_parse_json_object_v1") and EVALUATOR_VERSION=="shared_rag_all_gold_em_f1_v1")
assert all(SANITY.values()),{k:v for k,v in SANITY.items() if not v}
(DIRS["logs"]/"sanity_checks.json").write_text(json.dumps(SANITY,indent=2),encoding="utf-8")
print("ALL REQUIRED SANITY TESTS PASSED")
print(json.dumps(SANITY,indent=2))

In [ ]:
# Cell 16 - isolated resumable 16-question execution; gold appears only after prediction
CONFIG={
    "EXPERIMENT_ID":EXPERIMENT_ID,"seed":SEED,"model":MODEL_NAME,"temperature":TEMPERATURE,"top_p":TOP_P,"do_sample":DO_SAMPLE,
    "question_qids":SELECTED_QIDS,"selection_procedure":"RAG selected_questions.json when available, otherwise identical seed-42 stratified fallback","context_mode":CONTEXT_MODE,
    "official_image_resolution":{"records":len(images_raw),"resolved":len(IMAGE_ID_TO_PATH),"missing":len(images_raw)-len(IMAGE_ID_TO_PATH)},
    "answer_parser_version":ANSWER_PARSER_VERSION,"evaluator_version":EVALUATOR_VERSION,"prompt_version":PROMPT_VERSION,
    "dataset_paths":{k:str(v) for k,v in PATHS.items()},"image_reasoning_uses_title":IMAGE_REASONING_USES_TITLE,
    "full_corpus_retrieval":False,"hm_rag":False,"verifier":False,"re_retrieval":False,"enable_web":ENABLE_WEB,
    "context_caps":{"text_per_doc_chars":MAX_TEXT_CHARS_PER_DOC,"total_text_chars":MAX_TOTAL_TEXT_CHARS,"table_chars":MAX_TABLE_CHARS},
}
CONFIG_FINGERPRINT=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()
config_path=DIRS["logs"]/"config.json"
if config_path.exists():assert json.loads(config_path.read_text(encoding="utf-8"))==CONFIG,"Existing experiment ID has a different configuration"
else:config_path.write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")

def read_predictions(path):
    if not Path(path).exists():return []
    rows=[]
    with Path(path).open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try:rows.append(json.loads(line))
                except json.JSONDecodeError as e:raise ValueError(f"{path}:{no}: {e}") from e
    qids=[r["qid"] for r in rows]
    assert len(qids)==len(set(qids)),"Duplicate qids in prediction checkpoint"
    assert set(qids)<=set(SELECTED_QIDS),"Foreign/stale qids in this experiment checkpoint"
    assert all(r.get("experiment_id")==EXPERIMENT_ID and r.get("config_fingerprint")==CONFIG_FINGERPRINT for r in rows),"Checkpoint rows belong to a different experiment/configuration"
    return rows

def append_prediction(row):
    with PREDICTION_PATH.open("a",encoding="utf-8") as f:
        f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n"); f.flush(); os.fsync(f.fileno())

done={r["qid"] for r in read_predictions(PREDICTION_PATH)}
for no,item in enumerate(PROVIDED_INPUTS,1):
    if item.qid in done:continue
    print(f"\nMAMMQA PROVIDED CONTEXT {no}/16 | {item.qid}")
    prediction=run_mammqa(item,verbose=True)
    assert prediction.get("parsed_final_answer") is not None
    # Only now, after Stage III has produced the final prediction, materialize gold annotations.
    raw=DEV_BY_QID[item.qid]; ev=evaluation_only(raw)
    em,f1=em_f1(prediction["parsed_final_answer"],ev.gold_answers)
    prediction.update({"experiment_id":EXPERIMENT_ID,"config_fingerprint":CONFIG_FINGERPRINT,"gold_answers":list(ev.gold_answers),"gold_support_ids":list(ev.gold_support_ids),"gold_support_modalities":ev.gold_support_modalities,"exact_match":em,"token_f1":f1})
    append_prediction(prediction)

RESULTS=read_predictions(PREDICTION_PATH)
result_order={qid:i for i,qid in enumerate(SELECTED_QIDS)}
RESULTS=sorted(RESULTS,key=lambda r:result_order[r["qid"]])
result_qids=[r["qid"] for r in RESULTS]
assert len(RESULTS)==16 and len(result_qids)==len(set(result_qids))==16
assert set(result_qids)==set(SELECTED_QIDS)
assert all(not math.isnan(float(r["exact_match"])) and not math.isnan(float(r["token_f1"])) for r in RESULTS)
assert all(r["active_agent_count"]==2*len(r["available_modalities"])+1 for r in RESULTS)
image_rows=[r for r in RESULTS if "image" in r["available_modalities"]]
assert image_rows and all(r["runtime_trace"]["image_pixel_count"]>0 for r in image_rows)
SANITY["actual_run_image_pixels_sent_to_qwen"]=True
(DIRS["logs"]/"sanity_checks.json").write_text(json.dumps(SANITY,indent=2),encoding="utf-8")
audit_path=DIRS["logs"]/"qwen_audit.json"
prior=json.loads(audit_path.read_text(encoding="utf-8")) if audit_path.exists() else []
if QWEN_AUDIT or not audit_path.exists():audit_path.write_text(json.dumps(prior+QWEN_AUDIT,indent=2),encoding="utf-8")
print("16/16 PROVIDED-CONTEXT PREDICTIONS COMPLETE; all runtime assertions passed")

In [ ]:
# Cell 17 - failure analysis, required CSV/JSON outputs and final tables
def output_answer_matches(raw,golds):return em_f1(parse_final_answer(raw),golds)[0]==1

def classify_failure(r):
    if r["exact_match"]:return "correct"
    provided=set(r["provided_text_ids"]+r["provided_table_ids"]+r["provided_image_ids"]); gold_support=set(r.get("gold_support_ids",[]))
    if gold_support and not gold_support<=provided:return "provided_context_insufficient"
    normalized_raw=norm_answer(r.get("raw_aggregator_output",""))
    if any(norm_answer(g) and norm_answer(g) in normalized_raw for g in r["gold_answers"]):return "answer_parsing_failure"
    if any(output_answer_matches(x,r["gold_answers"]) for x in r.get("stage2",{}).values()):return "aggregation_failure"
    if any(output_answer_matches(x,r["gold_answers"]) for x in r.get("stage1",{}).values()):return "cross_modal_synthesis_failure"
    image_gold=any(r.get("gold_support_modalities",{}).get(i)=="image" for i in gold_support)
    if image_gold and "image" in r["available_modalities"]:return "vision_failure"
    if r.get("stage1"):return "expert_extraction_failure"
    return "other_reasoning_failure"

per_question=[]; failure_rows=[]
for no,r in enumerate(RESULTS,1):
    failure=classify_failure(r)
    row={"#":no,"qid":r["qid"],"modalities":"+".join(r["available_modalities"]),"gold":" | ".join(r["gold_answers"]),"prediction":r["parsed_final_answer"],"EM":r["exact_match"],"F1":r["token_f1"],"failure_type":failure,"active_agents":r["active_agent_count"],"text_sources":r["context_source_counts"]["text"],"table_sources":r["context_source_counts"]["table"],"image_sources":r["context_source_counts"]["image"]}
    per_question.append(row); failure_rows.append({"qid":r["qid"],"question":r["question"],"gold_answers":" | ".join(r["gold_answers"]),"prediction":r["parsed_final_answer"],"failure_type":failure,"modalities":"+".join(r["available_modalities"]),"active_agents":r["active_agent_count"]})

pdf=pd.DataFrame(per_question); fdf=pd.DataFrame(failure_rows)
pdf.to_csv(DIRS["metrics"]/"per_question.csv",index=False); fdf.to_csv(DIRS["metrics"]/"failure_analysis.csv",index=False)
correct=int(pdf["EM"].sum()); mean_f1=float(pdf["F1"].mean()); abstentions=sum(bool(r["abstain"]) for r in RESULTS); counts=Counter(fdf["failure_type"])
SUMMARY={"experiment_id":EXPERIMENT_ID,"questions_evaluated":len(RESULTS),"exact_correct":correct,"em_accuracy":correct/16,"mean_token_f1":mean_f1,"abstentions":abstentions,"retrieval_metrics":"N/A - provided context","failure_breakdown":dict(counts)}
(DIRS["metrics"]/"summary.json").write_text(json.dumps(SUMMARY,indent=2),encoding="utf-8")

print("\n| # | qid | modalities | gold | prediction | EM | F1 | failure_type | active agents |")
print(pdf[["#","qid","modalities","gold","prediction","EM","F1","failure_type","active_agents"]].to_markdown(index=False,floatfmt=".3f"))
print("\n================================================")
print("MAMMQA_PROVIDED_CONTEXT_16")
print("================================================")
print("Questions evaluated: 16 / 16")
print("Model: Qwen2.5-VL-7B-Instruct")
print("Seed: 42")
print("Context mode: PROVIDED_CONTEXT")
print("Official MMQA images: YES")
print("Full-corpus retrieval: NO")
print("HM-RAG: NO")
print("Verifier/re-retrieval: NO")
print(f"\nExact correct:\n{correct} / 16")
print(f"\nEM / Accuracy:\n{100*correct/16:.2f}%")
print(f"\nMean Token F1:\n{100*mean_f1:.2f}%")
print(f"\nAbstentions:\n{abstentions} / 16")
print("\nFailure breakdown:")
for key in ["expert_extraction_failure","cross_modal_synthesis_failure","vision_failure","aggregation_failure","provided_context_insufficient","answer_parsing_failure","other_reasoning_failure","correct"]:print(f"{key}: {counts.get(key,0)}")
print("\nRetrieval metrics: N/A - this is a provided-context reasoning baseline.")
print("Each exact answer changes EM by 6.25 percentage points.")
print(f"\nMAMMQA_Provided_Context_16 answered {correct}/16 questions exactly correctly, corresponding to {100*correct/16:.2f}% EM, with mean token F1 of {100*mean_f1:.2f}%. This experiment measures MAMMQA reasoning when question-associated MultiModalQA context is supplied directly. It does not evaluate full-corpus retrieval. The result is a controlled smoke-test reference and not a broader performance claim.")

In [ ]:
# Cell 18 - optional read-only comparison with an already completed RAG smoke run
RAG_SUMMARY_CANDIDATES=[
    Path("/kaggle/working/hm_mammqa/qa_smoke16_full_pipeline_seed42/metrics/summary.csv"),
]
if input_root.exists():RAG_SUMMARY_CANDIDATES += list(input_root.rglob("summary.csv"))
rag_summary_path=next((p for p in RAG_SUMMARY_CANDIDATES if p.is_file() and "qa_smoke16_full_pipeline_seed42" in str(p)),None)
if rag_summary_path:
    rag_df=pd.read_csv(rag_summary_path); full=rag_df[rag_df["Method"]=="FULL"]
    if len(full)==1:
        rag_em=float(full.iloc[0]["EM"]); rag_f1=float(full.iloc[0]["F1"])
        print("\nREAD-ONLY CONTROLLED COMPARISON")
        print(f"MAMMQA Provided Context: EM = {100*SUMMARY['em_accuracy']:.2f}%, F1 = {100*SUMMARY['mean_token_f1']:.2f}%")
        print(f"HM-RAG + MAMMQA: EM = {100*rag_em:.2f}%, F1 = {100*rag_f1:.2f}%")
        print(f"Difference (FULL_RAG_EM - PROVIDED_CONTEXT_EM): {100*(rag_em-SUMMARY['em_accuracy']):+.2f} points")
        print(f"Difference (FULL_RAG_F1 - PROVIDED_CONTEXT_F1): {100*(rag_f1-SUMMARY['mean_token_f1']):+.2f} points")
        print("Interpretation: provided-context MAMMQA receives question-associated evidence directly, whereas HM-RAG + MAMMQA must discover evidence. This is a reasoning reference / upper-bound-like control, not a pure retrieval ablation. The pure retrieval comparison remains Dense + MAMMQA versus HM-RAG + MAMMQA.")
else:
    print("RAG summary not present; no cross-experiment values were recomputed or altered.")